#### What is the Bronze Layer?

The Bronze layer takes **raw files from cloud storage** (CSV, JSON) and loads them into **Delta tables** with minimal processing.

Think of it like moving boxes from a delivery truck into a warehouse - you unpack nothing, you just store them in an organized place.

---

#### What we do in every Bronze notebook:

| Step | What it does | Why |
| --- | --- | --- |
| 1. Load Config | Import shared variables and helpers | Avoid repeating code |
| 2. Set Variables | Define source file path and target table name | Easy to change later |
| 3. Define Schema | Tell Spark what columns and types to expect | Faster reads, correct types |
| 4. Read File | Load CSV or JSON into a DataFrame | Get the data into Spark |
| 5. Add Metadata | Add `ingestion_timestamp` and `source_file` columns | Track when and where data came from |
| 6. Write Delta | Save as a Delta table in Unity Catalog | Makes data queryable with SQL |

---

#### Code Flow - Syntax Reference

---

##### 1. Load Configuration
```python
%run ../00-common/01.environment-config
%run ../00-common/02.bronze_helpers
```
Imports shared variables (`catalog_name`, `bronze_schema`, `loding_folder_path`) and the `add_ingestion_metadata()` helper function.

---

##### 2. Set Variables
```python
source_file = f'{loding_folder_path}/circuits.csv'
table_name = f'{catalog_name}.{bronze_schema}.circuits'
```
Define where the raw file is and where the Delta table will be saved.

---

##### 3. Define Schema

**Option A: StructType (for CSV and nested JSON)**
```python
from pyspark.sql.types import *

circuits_schema = StructType([
    StructField('circuitId', StringType(), True),
    StructField('circuitName', StringType(), True),
    StructField('lat', DoubleType(), True),
    StructField('long', DoubleType(), True),
    StructField('locality', StringType(), True),
    StructField('country', StringType(), True)
])
```

**Option B: DDL String (shorter, for flat JSON)**
```python
constructor_schema = 'constructorId STRING, name STRING, nationality STRING, url STRING'
```

**Option C: Nested StructType (for JSON with objects inside objects)**
```python
name_schema = StructType([
    StructField('givenName', StringType(), True),
    StructField('familyName', StringType(), True)
])

driver_schema = StructType([
    StructField('driverId', StringType(), True),
    StructField('name', name_schema),
    StructField('nationality', StringType(), True)
])
```
Used when a JSON field contains another object (like `name` containing `givenName` and `familyName`).

---

##### 4. Read File

**CSV files:**
```python
df = (
    spark.read
    .format('csv')
    .option('header', True)
    .schema(circuits_schema)
    .load(source_file)
)
```
- `format('csv')` - file type
- `option('header', True)` - first row is column names
- `.schema()` - apply our explicit schema
- `.load()` - path to the file

**JSON files (single file):**
```python
df = (
    spark.read
    .format('json')
    .schema(constructor_schema)
    .load(source_file)
)
```

**JSON files (folder with multiple files):**
```python
df = (
    spark.read
    .format('json')
    .schema(results_schema)
    .load(source_file)   # source_file points to a folder
)
```
Spark automatically reads ALL JSON files in the folder and combines them.

---

##### 5. Add Metadata Columns
```python
df_final = add_ingestion_metadata(df)
```
This helper function (from **`02.bronze_helpers`**) adds:
- `ingestion_timestamp` - when the data was loaded
- `source_file` - which file the row came from

---

##### 6. Write to Delta Table
```python
(
    df_final.write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)
```
- `format('delta')` - Delta format (versioning, fast queries)
- `mode('overwrite')` - replace the table completely each run
- `saveAsTable()` - register in Unity Catalog

**If schema changed since last run, add:**
```python
.option('overwriteSchema', 'true')
```

---

#### Notebooks in this folder:

| Notebook | File Type | Source | Key Notes |
| --- | --- | --- | --- |
| **`01) Ingest circuit files`** | CSV | Single file | StructType schema |
| **`02) Ingest race files`** | CSV | Single file | StructType schema with DateType |
| **`03) Ingest Constructor Files`** | JSON | Single file | DDL string schema (shorter syntax) |
| **`04) Ingest Driver File`** | JSON | Single file | Nested schema (name contains givenName + familyName) |
| **`05) Ingest Results Files`** | JSON | Folder (multiple files) | 14 columns, reads all files from folder |
| **`06) Ingest Sprints Files`** | JSON | Folder (multiple files) | Same structure as Results, multiLine option |